# Lab 2: Video Game Sales Analysis
### **Project Overview:**
The goal of this project is to build a distributed data pipeline using Spark RDDs. We are analyzing the Video Game Sales dataset, which contains records of games with sales greater than 100,000 copies. We aim to load, clean, and transform this data to find insights like total sales per genre and platform.

## Phase 1: Data Loading & Inspection
* Created a Spark session using local[*] to use all CPU cores.
* Used sc.textFile() to load the data into 4 partitions.
* Used the csv module to handle complex titles (titles containing commas).
* Removed the header and verified the data structure.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving vgsales.csv to vgsales (1).csv


In [ ]:
import csv
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("VideoGameAnalysis_Lab2") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
raw_rdd = sc.textFile("vgsales.csv", minPartitions=4)

In [ ]:
header = raw_rdd.first()
data_no_header = raw_rdd.filter(lambda row: row != header)

In [ ]:
def parse_csv(line):
    return next(csv.reader([line]))

split_rdd = data_no_header.map(parse_csv)

In [ ]:
print(f"Sample Records (First 3):")
for record in split_rdd.take(3):
    print(f" -> {record}")

Sample Records (First 3):
 -> ['1', 'Wii Sports', 'Wii', '2006', 'Sports', 'Nintendo', '41.49', '29.02', '3.77', '8.46', '82.74']
 -> ['2', 'Super Mario Bros.', 'NES', '1985', 'Platform', 'Nintendo', '29.08', '3.58', '6.81', '0.77', '40.24']
 -> ['3', 'Mario Kart Wii', 'Wii', '2008', 'Racing', 'Nintendo', '15.85', '12.88', '3.79', '3.31', '35.82']


* All values were initially loaded as Strings.

In [ ]:
total_records = split_rdd.count()
print(f"\nTotal records: {total_records}")


Total records: 16598


In [ ]:
na_count = split_rdd.filter(lambda row: "N/A" in row).count()
print(f"rows with missing values : {na_count}")

rows with missing values : 307


In [ ]:
actual_partitions = split_rdd.getNumPartitions()
print(f"Number of partitions used: {actual_partitions}")


Number of partitions used: 4


* By splitting 16,598 records into 4 partitions, Spark will process 4,150 records per partition.

##  Phase 2: Data Cleaning (RDD Transformations)
* Filtered out malformed rows that did not have exactly 11 columns.
* Trimmed extra spaces and removed rows with "N/A" values.
* Converted rows to Tuples to remove 307 duplicate records using distinct().
* Converted text columns (Rank, Year, Sales) into Integers and Floats.
* Standardized text to uppercase for consistency

In [ ]:
clean_rdd = split_rdd.filter(lambda row: len(row) == 11)

In [ ]:
clean_rdd = clean_rdd.map(lambda row: [col.strip() for col in row])

In [ ]:
clean_rdd = clean_rdd.filter(lambda row: all(col != "N/A" and col != "" for col in row))

In [ ]:
clean_rdd = clean_rdd.map(lambda row: tuple(row)).distinct()

In [ ]:
clean_rdd = clean_rdd.map(lambda row: [col.upper() if isinstance(col, str) else col for col in row])

In [ ]:
def convert_types(row_tuple):
    try:
        row = list(row_tuple)
        row[0] = int(row[0])    # rank
        row[3] = int(row[3])    # year
        row[6] = float(row[6])  # NAsales
        row[7] = float(row[7])  # EUsales
        row[8] = float(row[8])  # JPsales
        row[9] = float(row[9])  # othersales
        row[10] = float(row[10])# globalsales
        return row
    except (ValueError, IndexError):
        return None

In [ ]:
final_clean_rdd = clean_rdd.map(convert_types).filter(lambda x: x is not None)

In [ ]:
print(f"Total records after cleaning: {final_clean_rdd.count()}")
print("\nSample Cleaned Record (Proper Data Types):")
print(final_clean_rdd.take(1))

Total records after cleaning: 16291

Sample Cleaned Record (Proper Data Types):
[[3, 'MARIO KART WII', 'WII', 2008, 'RACING', 'NINTENDO', 15.85, 12.88, 3.79, 3.31, 35.82]]


## Phase 3: Core RDD Transformations  

### map() ➡ Transform each row
* Extract (Name, Global_Sales) from each row.
* map() transforms each row independently, full row → (game name, global sales)
* To transform each record by extracting only the fields we care about
(in this case: game name and global sales).
* This reduces data size and prepares it for analysis.

In [ ]:
name_global_sales = final_clean_rdd.map(
    lambda row: (row[1], row[10])
)

In [ ]:
name_global_sales.take(3)

[('MARIO KART WII', 35.82),
 ('GRAND THEFT AUTO: SAN ANDREAS', 20.81),
 ('SUPER MARIO WORLD', 20.61)]

### filter() ➡ Select specific rows
* Get only games with Global_Sales > 20 million
* filter() keeps rows that satisfy a condition


In [ ]:
top_selling_games = final_clean_rdd.filter(
    lambda row: row[10] > 20
)

In [ ]:
top_selling_games.take(3)

[[3,
  'MARIO KART WII',
  'WII',
  2008,
  'RACING',
  'NINTENDO',
  15.85,
  12.88,
  3.79,
  3.31,
  35.82],
 [18,
  'GRAND THEFT AUTO: SAN ANDREAS',
  'PS2',
  2004,
  'ACTION',
  'TAKE-TWO INTERACTIVE',
  9.43,
  0.4,
  0.41,
  10.57,
  20.81],
 [19,
  'SUPER MARIO WORLD',
  'SNES',
  1990,
  'PLATFORM',
  'NINTENDO',
  12.78,
  3.75,
  3.54,
  0.55,
  20.61]]

### distinct() ➡ Remove duplicates
* Used here to find unique gaming platforms in the dataset
* map() extracts platform column
* distinct() removes duplicates

In [ ]:
platforms = final_clean_rdd.map(lambda row: row[2]).distinct()

In [ ]:
platforms.take(10)

['SNES', 'GEN', 'GC', 'WS', 'PS4', 'XB', 'GB', 'PC', 'WIIU', '3DS']

### flatMap() ➡ One row → many rows
* To convert one record into multiple output records.
* We split game names into individual words
* map() would give list of words
* flatMap() flattens them into one sequence


In [ ]:
name_words = final_clean_rdd.flatMap(
    lambda row: row[1].split(" ")
)

In [ ]:
name_words.take(10)


['MARIO',
 'KART',
 'WII',
 'GRAND',
 'THEFT',
 'AUTO:',
 'SAN',
 'ANDREAS',
 'SUPER',
 'MARIO']

### sample() – Random sampling
* Take a random 10% of the dataset
* Does NOT scan full data logically

In [ ]:
sample_games = final_clean_rdd.sample(
    withReplacement=False,
    fraction=0.1,
    seed=42
)

In [ ]:
part = sample_games.count()
print(f"\nTotal records: {part}")


Total records: 1622


In [ ]:
sample_games.take(3)

[[3,
  'MARIO KART WII',
  'WII',
  2008,
  'RACING',
  'NINTENDO',
  15.85,
  12.88,
  3.79,
  3.31,
  35.82],
 [24,
  'GRAND THEFT AUTO V',
  'X360',
  2013,
  'ACTION',
  'TAKE-TWO INTERACTIVE',
  9.63,
  5.31,
  0.06,
  1.38,
  16.38],
 [45,
  'GRAND THEFT AUTO V',
  'PS4',
  2014,
  'ACTION',
  'TAKE-TWO INTERACTIVE',
  3.8,
  5.81,
  0.36,
  2.02,
  11.98]]

### repartition() ➡ Change number of partitions
* To change the number of partitions, which affects parallelism
and performance in distributed processing.
* repartition() reshuffles data
* Important for performance tuning


In [ ]:
before = final_clean_rdd.getNumPartitions()

In [ ]:
repartitioned_rdd = final_clean_rdd.repartition(6)

In [ ]:
after = repartitioned_rdd.getNumPartitions()

In [ ]:
before, after

(4, 6)

### Summary
| Operation   | Purpose                 |
| ----------- | ----------------------- |
| map         | Transform each row      |
| filter      | Select specific records |
| distinct    | Remove duplicates       |
| flatMap     | Split one row into many |
| sample      | Random subset           |
| repartition | Change parallelism      |


## Phase 4: Key-Value Operations

Used reduceByKey() to calculate the total global sales for each genre by summing the Global_Sales values of games that belong to the same genre.

groupByKey() to group game names by platform, allowing us to see all games released on each platform together.

countByKey() to count how many games exist in each genre, which helps identify the most common genres in the dataset.

sortByKey() to organize the results by key, making the aggregated data easier to read and analyze.

In [ ]:
genre_sales = final_clean_rdd.map(lambda row: (row[4], row[10])) \
                             .reduceByKey(lambda a, b: a + b)

print("Total Global Sales per Genre:")
for genre, sales in genre_sales.collect():
    print(genre, round(sales,2))

Total Global Sales per Genre:
SIMULATION 389.98
RACING 726.76
PUZZLE 242.21
FIGHTING 444.05
ADVENTURE 234.59
ACTION 1722.84
SPORTS 1309.24
PLATFORM 829.13
ROLE-PLAYING 923.83
SHOOTER 1026.2
MISC 789.87
STRATEGY 173.27


## reduceByKey() –> Total Global Sales per Genre

In [ ]:
platform_games = final_clean_rdd.map(lambda row: (row[2], row[1])) \
                                .groupByKey()

print("\nGames grouped by Platform (sample):")
for platform, games in platform_games.take(5):
    print(platform, list(games)[:5])   # show first 5 games only


Games grouped by Platform (sample):
SNES ['SUPER MARIO WORLD', 'DONKEY KONG COUNTRY', 'F-ZERO', 'CHRONO TRIGGER', 'SECRET OF MANA']
GEN ['SONIC THE HEDGEHOG 2', "STREET FIGHTER II': SPECIAL CHAMPION EDITION", 'SUPER STREET FIGHTER II', 'ECCO THE DOLPHIN', 'DRAGON SLAYER: THE LEGEND OF HEROES']
GC ["LUIGI'S MANSION", 'ANIMAL CROSSING', 'POKÉMON COLOSSEUM', 'MARIO PARTY 4', 'STAR WARS ROGUE LEADER: ROGUE SQUADRON II']
WS ['FINAL FANTASY II', 'DIGIMON ADVENTURE: ANODE TAMER', 'CHOCOBO NO FUSHIGI DUNGEON FOR WONDERSWAN', 'SUPER ROBOT TAISEN COMPACT 2 DAI-1-BU', 'FINAL FANTASY']
PS4 ['CALL OF DUTY: BLACK OPS 3', 'GRAND THEFT AUTO V', 'FIFA 16', 'FALLOUT 4', "UNCHARTED 4: A THIEF'S END"]


## groupByKey() –> Games Grouped by Platform

In [ ]:
genre_count = final_clean_rdd.map(lambda row: (row[4], 1)).countByKey()

print("\nNumber of Games per Genre:")
for genre, count in genre_count.items():
    print(genre, count)


Number of Games per Genre:
RACING 1225
ACTION 3251
PLATFORM 875
ROLE-PLAYING 1470
PUZZLE 570
SHOOTER 1282
SPORTS 2304
MISC 1686
FIGHTING 836
ADVENTURE 1274
STRATEGY 670
SIMULATION 848


## countByKey() –> Number of Games per Genre

In [ ]:
sorted_genre_sales = genre_sales.sortByKey()

print("\nSorted Genre Sales:")
for genre, sales in sorted_genre_sales.collect():
    print(genre, round(sales,2))


Sorted Genre Sales:
ACTION 1722.84
ADVENTURE 234.59
FIGHTING 444.05
MISC 789.87
PLATFORM 829.13
PUZZLE 242.21
RACING 726.76
ROLE-PLAYING 923.83
SHOOTER 1026.2
SIMULATION 389.98
SPORTS 1309.24
STRATEGY 173.27


## sortByKey() –> Sorted Total Sales by Genre

## **Phase 5: Partitioning & Performance**

Analyze number of partitions

In [ ]:
print("Partitions before:", final_clean_rdd.getNumPartitions())
repartitioned_rdd = final_clean_rdd.repartition(6)
print("Partitions after:", repartitioned_rdd.getNumPartitions())


Partitions before: 4
Partitions after: 6


Use Cache

Compare execution time before/after caching


*   Before caching, Spark recomputed the entire transformation pipeline from the raw dataset, resulting in a longer execution time of about 1.099 seconds.
*   After caching, Spark reused the stored intermediate results in memory, reducing recomputation and lowering the execution time to about 0.932 seconds, demonstrating improved efficiency.



In [ ]:
import time

#no caching
start = time.time()
genre_sales.collect()
print("Execution time (no cache):", time.time() - start)

# With caching
genre_sales.cache()
start = time.time()
genre_sales.collect()
print("Execution time (cached):", time.time() - start)


Execution time (no cache): 1.0990545749664307
Execution time (cached): 0.932213306427002


 coalesce

In [ ]:
coalesced_rdd = final_clean_rdd.coalesce(2)
print("Partitions after coalesce:", coalesced_rdd.getNumPartitions())


Partitions after coalesce: 2


**Discuss performance observations**

*   The dataset initially had 4 partitions, and after applying repartition(6) it increased to 6 partitions, showing Spark’s ability to redistribute data for parallel processing.
*  Repartitioning improves parallelism but introduces shuffle overhead, which may not be efficient for smaller datasets.


*   The execution time without caching was approximately 1.099 seconds, as Spark recomputed the entire transformation pipeline.
*  The execution time with caching dropped to 0.932 seconds, since Spark reused stored results in memory instead of recomputing.



*   Caching is especially beneficial when the same RDD is accessed multiple times, reducing redundant computation.
*   Using coalesce(2) reduced the dataset to 2 partitions without shuffle, which is efficient for downsizing but decreases parallelism



*  repartition is useful for increasing parallelism, coalesce is efficient for reducing partitions, and cache improves performance by avoiding recomputation.










